# Sesión 10 · Combina tablas sin perder el control

Badge 4 · Manipulación de datos

Integra archivos y catálogos con claves, índices y comprobaciones explícitas.

150 minutos de clase; tarea de 45–60 minutos. Ejecuta en orden.

## concat: apilar archivos y alinear índices

concat con axis=0 reúne filas y alinea columnas por nombre. ignore_index=True crea un índice consecutivo, pero no elimina registros duplicados. Agrega una columna de origen antes de combinar para conservar trazabilidad.

Con axis=1 se agregan columnas alineadas por etiqueta del índice; no por la posición que ves en pantalla. Una etiqueta ausente en uno de los objetos produce un faltante. Revisa los esquemas y concatena una lista de tablas de una sola vez.

In [1]:
import pandas as pd
a = pd.DataFrame({"Producto": ["A", "B"], "Unidades": [2, 3]})
b = pd.DataFrame({"Producto": ["A"], "Unidades": [4]})
unidas = pd.concat([a.assign(Origen="enero"), b.assign(Origen="febrero")], ignore_index=True)
print(unidas)
print(unidas["Unidades"].sum())

  Producto  Unidades   Origen
0        A         2    enero
1        B         3    enero
2        A         4  febrero
9


**Qué observar:** Tres registros; total 9 unidades.

**Recuerda:** concat une estructuras sobre un eje; no busca claves de negocio como merge.

## merge: elegir qué registros conservar

Una unión inner conserva claves presentes en ambos lados; left conserva todas las filas izquierdas; outer conserva claves de ambos lados. on nombra la clave. Si sus nombres difieren, usa left_on y right_on.

indicator crea _merge para identificar coincidencias y ausencias. Pandas puede hacer coincidir claves nulas entre sí: revisa y trata las claves faltantes antes de unir, especialmente si vienes de SQL.

In [2]:
import pandas as pd
v = pd.DataFrame({"id": [1, 2, 3], "Unidades": [2, 4, 1]})
c = pd.DataFrame({"id": [1, 2, 4], "Categoria": ["A", "B", "C"]})
for tipo in ["inner", "left", "outer"]:
    u = v.merge(c, on="id", how=tipo, validate="one_to_one", indicator=True)
    print(tipo, len(u))
print(v.merge(c, on="id", how="left", indicator=True))

inner 2
left 3
outer 4
   id  Unidades Categoria     _merge
0   1         2         A       both
1   2         4         B       both
2   3         1       NaN  left_only


**Qué observar:** inner: 2 filas; left: 3; outer: 4. La clave 3 queda left_only.

**Recuerda:** Elige how según el universo que deseas conservar, no según el resultado que se vea más completo.

## Cardinalidad: la defensa contra totales inflados

Una relación many_to_one admite muchas ventas para un solo producto de catálogo. validate verifica esa expectativa y falla si el catálogo tiene claves duplicadas. Un many_to_many puede producir todas las combinaciones de coincidencias.

Compara cantidad de filas y suma de unidades antes y después de enriquecer ventas. Usa suffixes si ambos lados contienen columnas de igual nombre y selecciona explícitamente cuál representa el indicador correcto.

In [3]:
import pandas as pd
v = pd.DataFrame({"id": [1, 1, 2], "Unidades": [2, 3, 4]})
c = pd.DataFrame({"id": [1, 2], "Nombre": ["A", "B"]})
u = v.merge(c, on="id", how="left", validate="many_to_one")
assert len(u) == len(v)
assert u.Unidades.sum() == v.Unidades.sum()
print(u.Unidades.sum())

9


**Qué observar:** Total conservado: 9.

**Recuerda:** No quites validate para silenciar un error: investiga primero las claves.

## Reto · Archivos mensuales

Concatena cantidades [2,3] y [4], conservando origen.

**Criterio:** 3 filas y 9 unidades.

In [ ]:
# Tu solución aquí.

## Reto · Una clave sin catálogo

Une claves [1,2] con un catálogo que solo tiene 1. Identifica la ausencia.

**Criterio:** La clave 2 no tiene correspondencia.

In [ ]:
# Tu solución aquí.

## Tarea · Laboratorio · merge y concat

Integra archivos y catálogos con claves, índices y comprobaciones explícitas.

1. Divide ventas.csv en dos archivos y vuelve a unirlos con concat y columna de origen.
2. Crea un catálogo y enriquece con merge many_to_one.
3. Agrega deliberadamente una clave repetida al catálogo y documenta el error de validación.
4. Entrega un control de filas, unidades y claves sin correspondencia.

In [ ]:
# Desarrolla tu tarea y explica los resultados.